# DeepSpeed pipeline (1F1B đồng bộ) — VGG-16 / Tiny-ImageNet trên Kaggle T4 ×2
Thiết kế: paper_notes §26. Yêu cầu: Accelerator = **GPU T4 x2**, Internet = **On**, *Add Input* dataset `tin-cache` (5 file .npy/json; hoặc dataset gốc `tiny-imagenet-200`).
Chạy lần lượt từ trên xuống (chạy nền Save & Run All cũng được): setup → dữ liệu → cài DeepSpeed → quick (~5 phút) → 80 epoch (~2.5–3 h) → zip.
Không cần check D1–D4 (chúng kiểm tra runtime của mình, không phải DeepSpeed). Dán output của bản quick cho agent.

## 1. Repo + cài đặt + môi trường

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), '| nccl', torch.cuda.nccl.version())"
!find /kaggle/input -maxdepth 6 \( -name wnids.txt -o -name stats.json \)

## 2. Dữ liệu (cache `tin-cache` nếu có, nếu không thì giải mã ~3–5 phút)

In [ ]:
!python -c "from timeprest.config import load_config; from timeprest.data import build_datasets; import json, os; c = load_config('configs/tin_base.yaml'); tr, te = build_datasets(c['data'], 200, 0); print('train', len(tr), 'val(test)', len(te), 'sample', tuple(tr[0][0].shape), '| cache', tr.cache_dir); print(json.load(open(os.path.join(tr.cache_dir, 'stats.json'))))"

## 3. Cài DeepSpeed

In [ ]:
!pip install -q deepspeed
!python -c "import deepspeed, torch; print('deepspeed', deepspeed.__version__, '| torch', torch.__version__)"

## 4. Chạy thử (10k ảnh, 3 epoch): loss phải giảm, in thời gian/epoch

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.deepspeed_train --config configs/tin_deepspeed_quick.yaml

## 5. Chạy 80 epoch (`--resume` nếu bị ngắt trong cùng phiên)

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.deepspeed_train --config configs/tin_deepspeed.yaml --resume

## 6. Đóng gói kết quả

In [ ]:
!cd /kaggle/working && zip -qr /kaggle/working/results_tin_deepspeed.zip runs/tin_vgg16_deepspeed/*.csv runs/tin_vgg16_deepspeed/*.json runs/tin_deepspeed_quick/*.csv
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_tin_deepspeed.zip'))